Ejercicio 1. Crear tabla de prueba e insertar reservas canceladas de City Hotel

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_hotel.tb_reservas_prueba
USING DELTA
AS
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE 1 = 0;

INSERT INTO cursodataacademy.proyecto_hotel.tb_reservas_prueba
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE hotel = 'City Hotel'
  AND is_canceled = 1;

SELECT COUNT(*) AS total_reservas_insertadas
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba;

La tabla de prueba contiene únicamente reservas canceladas correspondientes a City Hotel. Esto permite realizar modificaciones sin alterar la tabla original.

Ejercicio 2. Actualizar babies a cero

In [0]:
%sql
UPDATE cursodataacademy.proyecto_hotel.tb_reservas_prueba
SET babies = 0;

SELECT babies, COUNT(*) AS cantidad
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
GROUP BY babies;

Se corrige el campo babies en la tabla de prueba, dejando el valor en cero para todos los registros, sin modificar la información original.

In [0]:
Ejercicio 3. Crear la fecha de llegada en formato DD-Mes-YYYY

In [0]:
%sql
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'),
        '-',
        arrival_date_month,
        '-',
        CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

La fecha de llegada se presenta en un formato más legible, facilitando su interpretación y presentación en los reportes.

Ejercicio 4. Filtrar canales de distribución que comienzan con T

In [0]:
%sql
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel LIKE 'T%';

Los indicadores permiten conocer el volumen total de reservas, la demanda de solicitudes especiales y el precio prom por noche del conjunto de datos.

Ejercicio 6. Eliminar reservas con ADR igual a cero de la tabla de prueba

In [0]:
%sql
DELETE FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
WHERE adr = 0.0;

SELECT COUNT(*) AS reservas_restantes
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba;

Ejercicio 7. Cantidad de reservas por canal de distribución

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS cantidad_reservas
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

Ejercicio 8. Clasificar las reservas según el precio ADR

In [0]:
%sql
SELECT
    adr,
    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        WHEN adr > 150 THEN 'Premium'
        ELSE 'Sin clasificación'
    END AS categoria_precio
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Ejercicio 9. Calcular la fecha en que se realizó la reserva

In [0]:
%sql
WITH reservas AS (
    SELECT
        *,
        DATE_ADD(
            TO_DATE(
                CONCAT('01-', arrival_date_month, '-', arrival_date_year),
                'dd-MMMM-yyyy'
            ),
            CAST(arrival_date_day_of_month - 1 AS INT)
        ) AS arrival_date
    FROM cursodataacademy.proyecto_hotel.hotel_bookings
)
SELECT
    arrival_date,
    lead_time,
    DATE_SUB(arrival_date, CAST(lead_time AS INT)) AS booking_date
FROM reservas;

Se obtiene una fecha estimada de creación de cada reserva restando los días de anticipación a la fecha de llegada.

Ejercicio 10. Calcular meses de anticipación redondeados hacia arriba


In [0]:
%sql
SELECT
    lead_time,
    CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

El indicador permite clasificar la anticipación de las reservas en meses aproximados, facilitando el análisis del tiempo transcurrido entre la reserva y la llegada